In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
train_dataset = pd.read_csv("/teamspace/studios/this_studio/Zindi_Clinical_Reasoning_Challenge/data/data/cleaned_train2.csv")
train_dataset.shape

(300, 12)

In [3]:
test_dataset = pd.read_csv("/teamspace/studios/this_studio/Zindi_Clinical_Reasoning_Challenge/data/data/cleaned_test2.csv")
test_dataset.shape

(100, 7)

In [4]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_dataset)
test_dataset = Dataset.from_pandas(test_dataset)

In [5]:
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
    num_rows: 300
})
Dataset({
    features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel'],
    num_rows: 100
})


In [6]:
# split the train dataset into train and validation sets
train_dataset = train_dataset.train_test_split(test_size=0.2)
print(train_dataset)

DatasetDict({
    train: Dataset({
        features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
        num_rows: 240
    })
    test: Dataset({
        features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
        num_rows: 60
    })
})


In [7]:
print(train_dataset)

DatasetDict({
    train: Dataset({
        features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
        num_rows: 240
    })
    test: Dataset({
        features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
        num_rows: 60
    })
})


In [14]:
train_dataset["train"]["clinician"]

[' a 27 year old female with complaints of back pain abdominal pain headache for 3 days vitals normal management of the patient admit administer analgesics for pain and headache administer intravenous fluids encourage rest and posture to reduce pain alert physician imo on duty for review ensure intravenous access for drawing of samples and fluids investigations complete blood count urea creatinine and electrolytes liver function tests blood gas analysis urinalysis pregnancy diagnostic test blood slide for malaria radiological tests abdominal ultrasound spine mri ct endoscopy                                                                                                                                                                                                                                                                                                                                                                                                                                   

### Tokenize the Dataset

In [8]:
import torch
import numpy as np
from transformers import (
    BartForConditionalGeneration, 
    BartTokenizer, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import evaluate

In [9]:
print("=== BART MODEL SETUP ===")

# 1. Load BART model and tokenizer
print("Loading BART-base model (139M parameters)...")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

print(f"Model parameters: {model.num_parameters():,}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

=== BART MODEL SETUP ===
Loading BART-base model (139M parameters)...


config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model parameters: 139,420,416
Tokenizer vocab size: 50265


In [10]:
# 2. Configure tokenizer for your task
# BART already has proper special tokens, no need to add new ones
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")
print(f"BOS token: {tokenizer.bos_token}")

Pad token: <pad>
EOS token: </s>
BOS token: <s>


In [11]:
# 3. Data preprocessing function for BART
def preprocess_clinical_data_for_bart(examples):
    """Preprocess clinical data for BART training"""
    
    # Format inputs for BART (more natural than T5)
    inputs = []
    targets = []
    
    for i in range(len(examples['prompt'])):
        # Create more natural input format for BART
        county = examples['county'][i]
        health_level = examples['health_level'][i] 
        experience = examples['years_of_experience'][i]
        prompt = examples['prompt'][i]
        
        # BART input format (more conversational)
        input_text = f"Clinical Case Analysis: A nurse with {experience} years of experience working in {health_level} in {county} county reports: {prompt} Please provide clinical assessment and management recommendations."
        
        # Target remains the same
        target_text = examples['clinician'][i] if 'clinician' in examples else ""
        
        inputs.append(input_text)
        targets.append(target_text)
    
    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=512,  # BART can handle longer sequences
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    
    # Tokenize targets
    labels = tokenizer(
        targets,
        max_length=256,  # Reasonable output length
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    
    # BART uses labels, not decoder_input_ids
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

In [12]:
# 4. Test preprocessing with a sample
print("\n=== TESTING BART PREPROCESSING ===")
sample_data = {
    'county': ['uasin gishu'],
    'health_level': ['national referral hospitals'],
    'years_of_experience': [18.0],
    'prompt': ['67 year old male with cough, blood-stained sputum, difficulty breathing, night sweats'],
    'clinician': ['Patient presents with hemoptysis and respiratory symptoms. Differential diagnosis includes tuberculosis, lung cancer. Recommend chest X-ray, sputum analysis, oxygen therapy.']
}

sample_processed = preprocess_clinical_data_for_bart(sample_data)
print(f"Input shape: {sample_processed['input_ids'].shape}")
print(f"Labels shape: {sample_processed['labels'].shape}")

# Decode to check formatting
sample_input = tokenizer.decode(sample_processed['input_ids'][0], skip_special_tokens=True)
sample_target = tokenizer.decode(sample_processed['labels'][0], skip_special_tokens=True)

print(f"\nFormatted input: {sample_input[:200]}...")
print(f"Target output: {sample_target[:200]}...")


=== TESTING BART PREPROCESSING ===
Input shape: torch.Size([1, 57])
Labels shape: torch.Size([1, 37])

Formatted input: Clinical Case Analysis: A nurse with 18.0 years of experience working in national referral hospitals in uasin gishu county reports: 67 year old male with cough, blood-stained sputum, difficulty breath...
Target output: Patient presents with hemoptysis and respiratory symptoms. Differential diagnosis includes tuberculosis, lung cancer. Recommend chest X-ray, sputum analysis, oxygen therapy....


In [13]:
# 5. BART training arguments (optimized for clinical task)
bart_training_args = Seq2SeqTrainingArguments(
    output_dir="./clinical_bart_model",
    eval_strategy="epoch",
    
    # BART-optimized learning rate
    learning_rate=3e-5,  # BART typically uses higher LR than T5
    
    # Batch settings
    per_device_train_batch_size=4,  # BART is more efficient
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16
    
    # Training schedule
    num_train_epochs=3,
    warmup_steps=100,
    weight_decay=0.01,
    
    # Generation settings
    predict_with_generate=True,
    generation_max_length=256,
    generation_num_beams=4,
    
    # Efficiency settings
    fp16=True,  # BART works well with mixed precision
    dataloader_pin_memory=True,
    save_total_limit=2,
    logging_steps=10,
    save_strategy="epoch",
    
    # Stability
    max_grad_norm=1.0,
    remove_unused_columns=False,
)

In [14]:
# 6. Data collator for BART
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)

In [15]:
# 7. Metrics for evaluation
metric = evaluate.load("rouge")

def compute_bart_metrics(eval_preds):
    """Compute metrics for BART training"""
    preds, labels = eval_preds
    
    if preds is None or labels is None:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "rougeLsum": 0.0}
    
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Replace -100 in labels with pad token
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Clean up text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    # Compute ROUGE
    try:
        result = metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=True
        )
        
        # Print sample for monitoring
        print(f"\nSample BART generation:")
        print(f"Pred: {decoded_preds[0][:150]}...")
        print(f"Target: {decoded_labels[0][:150]}...")
        
        return result
    except Exception as e:
        print(f"Metrics error: {e}")
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "rougeLsum": 0.0}

print(f"\n=== BART SETUP COMPLETE ===")
print(f"Model: BART-base ({model.num_parameters():,} parameters)")
print(f"Ready for training with optimized settings for clinical reasoning")
print(f"Next steps:")
print(f"1. Process your data: processed_data = preprocess_clinical_data_for_bart(your_data)")
print(f"2. Create trainer with BART model")
print(f"3. Start training")


=== BART SETUP COMPLETE ===
Model: BART-base (139,420,416 parameters)
Ready for training with optimized settings for clinical reasoning
Next steps:
1. Process your data: processed_data = preprocess_clinical_data_for_bart(your_data)
2. Create trainer with BART model
3. Start training


In [16]:
train_dataset

DatasetDict({
    train: Dataset({
        features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
        num_rows: 240
    })
    test: Dataset({
        features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
        num_rows: 60
    })
})

In [17]:
test_dataset

Dataset({
    features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel'],
    num_rows: 100
})

In [18]:
# 1. Convert to datasets
train_data = train_dataset["train"]
test_data = train_dataset["test"]
    
print(train_data)
print(test_data)

Dataset({
    features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
    num_rows: 240
})
Dataset({
    features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
    num_rows: 60
})


In [19]:
# Assuming you have your data in train_data and test_data format
def setup_bart_training(train_data, test_data):
    """Complete setup for BART training"""
    
    print("=== SETTING UP BART TRAINING PIPELINE ===")
    
    # # 1. Convert to datasets
    # train_dataset = train_dataset["train"]
    # test_dataset = train_dataset["test"]
    
    # print(f"Training samples: {len(train_dataset)}")
    # print(f"Test samples: {len(test_dataset)}")
    
    # 2. Preprocess data
    print("Preprocessing data for BART...")
    
    def preprocess_function(examples):
        return preprocess_clinical_data_for_bart(examples)
    
    # Apply preprocessing
    tokenized_train = train_data.map(
        preprocess_function,
        batched=True,
        remove_columns=train_data.column_names
    )
    
    tokenized_test = test_data.map(
        preprocess_function,
        batched=True,
        remove_columns=test_data.column_names
    )
    
    print("✅ Data preprocessing complete")
    
    # 3. Create trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=bart_training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_bart_metrics,
    )
    
    print("✅ BART trainer created")
    print("\nReady to train! Run: trainer.train()")
    
    return trainer, tokenized_train, tokenized_test

In [20]:
train_data

Dataset({
    features: ['master_index', 'county', 'health_level', 'years_of_experience', 'prompt', 'nursing_competency', 'clinical_panel', 'clinician', 'gpt4.0', 'llama', 'gemini', 'ddx_snomed'],
    num_rows: 240
})

In [21]:
trainer, train_tokens, test_tokens = setup_bart_training(train_data, test_data)

=== SETTING UP BART TRAINING PIPELINE ===
Preprocessing data for BART...


Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

✅ Data preprocessing complete


/tmp/ipykernel_4718/421710174.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


✅ BART trainer created

Ready to train! Run: trainer.train()


In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,3.279600,2.623428,0.270602,0.102193,0.191842,0.191551
2,2.675800,2.348943,0.285940,0.116513,0.203947,0.204006



Sample BART generation:
Pred: Clinical Case Analysis: A nurse with 38.0 years of experience working in  health centres   மனளி (2 years) in uasin gishu county in kenya a 48 year old...
Target: summary a 48 year old male with a hx of dizziness and frequent falls on warfarin tablets due to atrial fibrillation on exam noted small bruises on lef...


/teamspace/studios/this_studio/.conda/lib/python3.11/site-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



Sample BART generation:
Pred: Clinical Case Analysis: A nurse with 38.0 years of experience working in  health centres a 48 year old male client with complaints of dizziness and fa...
Target: summary a 48 year old male with a hx of dizziness and frequent falls on warfarin tablets due to atrial fibrillation on exam noted small bruises on lef...


TypeError: sequence item 70: expected str instance, NoneType found

In [ ]:
# Quick test function for BART
def test_bart_generation(model, tokenizer, sample_text):
    """Quick test of BART generation"""
    
    print(f"=== TESTING BART GENERATION ===")
    
    # Format input
    test_input = f"Clinical Case Analysis: {sample_text} Please provide clinical assessment and management recommendations."
    
    # Tokenize
    inputs = tokenizer(test_input, return_tensors="pt", max_length=512, truncation=True)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )
    
    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Input: {test_input[:100]}...")
    print(f"Generated: {generated_text}")
    
    return generated_text

# Example usage:
print("\nTo use this pipeline:")

print("1. trainer, train_tokens, test_tokens = setup_bart_training(your_train_data, your_test_data)")
print("2. trainer.train()")
print("3. test_bart_generation(model, tokenizer, 'patient has fever and cough')")

# Test with fresh BART model
sample_clinical_text = "A 45-year-old patient presents with chest pain and shortness of breath"
test_result = test_bart_generation(model, tokenizer, sample_clinical_text)